# PM2.5 Temporal Patterns Analysis (Japan Standard Time)

Comprehensive temporal analysis of actual PM2.5 concentrations for presentation
All times converted from UTC to JST (UTC+9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy import signal
import pytz
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10

## Load PM2.5 Data and Convert to JST

In [ ]:
print("Loading PM2.5 enriched dataset...")
data_path = '/Users/vojtech/Code/Bard89/smoglens-data/pm25_enriched_2023_2025_v4_20250830_222050.csv'

df = pd.read_csv(data_path, 
                 parse_dates=['timestamp'],
                 usecols=['timestamp', 'hex7_id', 'pm25_ugm3_mean', 'temperature_c_mean', 
                         'humidity_pct_mean', 'avg_traffic_volume'])

# Convert UTC to JST (Japan Standard Time = UTC+9)
print("Converting to Japan Standard Time (JST)...")
jst = pytz.timezone('Asia/Tokyo')
df['timestamp_jst'] = df['timestamp'].dt.tz_convert(jst)

# Extract time features in JST
df['hour'] = df['timestamp_jst'].dt.hour
df['day_of_week'] = df['timestamp_jst'].dt.dayofweek
df['month'] = df['timestamp_jst'].dt.month
df['year'] = df['timestamp_jst'].dt.year
df['date'] = pd.to_datetime(df['timestamp_jst'].dt.date)
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

print(f"Loaded {len(df):,} records")
print(f"Date range (JST): {df['timestamp_jst'].min()} to {df['timestamp_jst'].max()}")
print(f"PM2.5 coverage: {df['pm25_ugm3_mean'].notna().mean():.1%}")

## 1. Daily (24-hour) Patterns in JST

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

hourly_stats = df.groupby('hour')['pm25_ugm3_mean'].agg(['mean', 'std', 'median', 'count'])
hourly_stats['sem'] = hourly_stats['std'] / np.sqrt(hourly_stats['count'])

ax = axes[0, 0]
ax.plot(hourly_stats.index, hourly_stats['mean'], 'b-', linewidth=2.5, label='Mean')
ax.fill_between(hourly_stats.index, 
                hourly_stats['mean'] - hourly_stats['sem'],
                hourly_stats['mean'] + hourly_stats['sem'],
                alpha=0.3, color='blue', label='±SEM')
ax.plot(hourly_stats.index, hourly_stats['median'], 'r--', linewidth=1.5, label='Median')
ax.set_xlabel('Hour of Day (JST)', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('24-Hour PM2.5 Cycle (Japan Standard Time)', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 24, 3))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

peak_hour = hourly_stats['mean'].idxmax()
low_hour = hourly_stats['mean'].idxmin()
ax.annotate(f'Peak: {peak_hour}:00 JST\n({hourly_stats.loc[peak_hour, "mean"]:.1f} μg/m³)',
           xy=(peak_hour, hourly_stats.loc[peak_hour, 'mean']),
           xytext=(peak_hour+2, hourly_stats.loc[peak_hour, 'mean']+0.5),
           arrowprops=dict(arrowstyle='->', color='red', alpha=0.6))
ax.annotate(f'Low: {low_hour}:00 JST\n({hourly_stats.loc[low_hour, "mean"]:.1f} μg/m³)',
           xy=(low_hour, hourly_stats.loc[low_hour, 'mean']),
           xytext=(low_hour-4, hourly_stats.loc[low_hour, 'mean']-0.5),
           arrowprops=dict(arrowstyle='->', color='green', alpha=0.6))

ax = axes[0, 1]
weekday_hourly = df[df['is_weekend']==0].groupby('hour')['pm25_ugm3_mean'].mean()
weekend_hourly = df[df['is_weekend']==1].groupby('hour')['pm25_ugm3_mean'].mean()
ax.plot(weekday_hourly.index, weekday_hourly.values, 'b-', linewidth=2, label='Weekday', marker='o', markersize=4)
ax.plot(weekend_hourly.index, weekend_hourly.values, 'r-', linewidth=2, label='Weekend', marker='s', markersize=4)
ax.set_xlabel('Hour of Day (JST)', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Weekday vs Weekend Diurnal Pattern', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 24, 3))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
ax.axvspan(7, 9, alpha=0.1, color='orange', label='Morning rush')
ax.axvspan(17, 19, alpha=0.1, color='orange', label='Evening rush')

ax = axes[1, 0]
hourly_by_year = df.groupby(['year', 'hour'])['pm25_ugm3_mean'].mean().unstack(level=0)
for year in hourly_by_year.columns:
    ax.plot(hourly_by_year.index, hourly_by_year[year], marker='o', markersize=3, label=f'{year}')
ax.set_xlabel('Hour of Day (JST)', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Diurnal Pattern by Year', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 24, 3))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')

ax = axes[1, 1]
hourly_traffic = df.groupby('hour')['avg_traffic_volume'].mean()
hourly_pm25 = df.groupby('hour')['pm25_ugm3_mean'].mean()
ax2 = ax.twinx()
l1 = ax.plot(hourly_pm25.index, hourly_pm25.values, 'b-', linewidth=2.5, label='PM2.5')
l2 = ax2.plot(hourly_traffic.index, hourly_traffic.values, 'r-', linewidth=2.5, label='Traffic')
ax.set_xlabel('Hour of Day (JST)', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', color='b', fontsize=11)
ax2.set_ylabel('Traffic Volume', color='r', fontsize=11)
ax.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
ax.set_title('PM2.5 vs Traffic Diurnal Patterns (JST)', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 24, 3))
ax.grid(True, alpha=0.3)
lines = l1 + l2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right')

plt.suptitle('Daily PM2.5 Patterns Analysis (Japan Standard Time)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graphs/temporal/daily_patterns_jst.png', dpi=300, bbox_inches='tight')
print("Saved: graphs/temporal/daily_patterns_jst.png")
plt.show()

## 2. Weekly Patterns

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow_stats = df.groupby('day_of_week')['pm25_ugm3_mean'].agg(['mean', 'std', 'median', 'count'])
dow_stats['sem'] = dow_stats['std'] / np.sqrt(dow_stats['count'])

ax = axes[0, 0]
bars = ax.bar(dow_stats.index, dow_stats['mean'], 
              yerr=dow_stats['sem'], capsize=5,
              color=['#3498DB']*5 + ['#2ECC71']*2,
              edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Day of Week', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Weekly PM2.5 Pattern', fontsize=12, fontweight='bold')
ax.set_xticks(range(7))
ax.set_xticklabels(dow_names)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=dow_stats['mean'].mean(), color='red', linestyle='--', alpha=0.5, label=f'Mean: {dow_stats["mean"].mean():.1f}')

for i, (bar, val) in enumerate(zip(bars, dow_stats['mean'])):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
           f'{val:.1f}', ha='center', fontsize=9)
ax.legend()

ax = axes[0, 1]
weekend_comparison = df.groupby('is_weekend')['pm25_ugm3_mean'].agg(['mean', 'std', 'count'])
weekend_comparison['sem'] = weekend_comparison['std'] / np.sqrt(weekend_comparison['count'])
bars = ax.bar(['Weekday', 'Weekend'], weekend_comparison['mean'],
              yerr=weekend_comparison['sem'], capsize=8,
              color=['#3498DB', '#2ECC71'], edgecolor='black', linewidth=2, alpha=0.8)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Weekday vs Weekend Comparison', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, weekend_comparison['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
           f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

diff = weekend_comparison['mean'][0] - weekend_comparison['mean'][1]
pct_diff = diff / weekend_comparison['mean'][0] * 100
ax.text(0.5, max(weekend_comparison['mean']) * 0.5, 
       f'Difference: {diff:.2f} μg/m³\n({pct_diff:.1f}% lower on weekends)',
       ha='center', transform=ax.transData,
       bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))

ax = axes[1, 0]
dow_by_year = df.groupby(['year', 'day_of_week'])['pm25_ugm3_mean'].mean().unstack(level=0)
x = np.arange(7)
width = 0.25
for i, year in enumerate(dow_by_year.columns):
    ax.bar(x + i*width, dow_by_year[year], width, label=f'{year}')
ax.set_xlabel('Day of Week', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Weekly Pattern by Year', fontsize=12, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(dow_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1, 1]
daily_ts = df.groupby('date')['pm25_ugm3_mean'].mean()
weekly_rolling = daily_ts.rolling(window=7, center=True).mean()
ax.plot(daily_ts.index, daily_ts.values, alpha=0.3, linewidth=0.5, color='gray', label='Daily')
ax.plot(weekly_rolling.index, weekly_rolling.values, linewidth=2, color='blue', label='7-day MA')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Weekly Moving Average Trend', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Weekly PM2.5 Patterns Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graphs/temporal/weekly_patterns.png', dpi=300, bbox_inches='tight')
print("Saved: graphs/temporal/weekly_patterns.png")
plt.show()

## 3. Monthly and Seasonal Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_stats = df.groupby('month')['pm25_ugm3_mean'].agg(['mean', 'std', 'median', 'count'])
monthly_stats['sem'] = monthly_stats['std'] / np.sqrt(monthly_stats['count'])

ax = axes[0, 0]
bars = ax.bar(monthly_stats.index, monthly_stats['mean'],
              yerr=monthly_stats['sem'], capsize=5,
              color=plt.cm.coolwarm(np.linspace(0.2, 0.8, 12)),
              edgecolor='black', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Monthly PM2.5 Averages', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names, rotation=45)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=monthly_stats['mean'].mean(), color='red', linestyle='--', alpha=0.5)

peak_month = monthly_stats['mean'].idxmax()
low_month = monthly_stats['mean'].idxmin()
ax.annotate(f'Peak: {month_names[peak_month-1]}', xy=(peak_month, monthly_stats.loc[peak_month, 'mean']),
           xytext=(peak_month, monthly_stats.loc[peak_month, 'mean'] + 0.5),
           arrowprops=dict(arrowstyle='->', color='red', alpha=0.6))

ax = axes[0, 1]
df['season'] = df['month'].map(lambda x: 'Winter' if x in [12, 1, 2]
                               else 'Spring' if x in [3, 4, 5]
                               else 'Summer' if x in [6, 7, 8]
                               else 'Fall')
season_stats = df.groupby('season')['pm25_ugm3_mean'].agg(['mean', 'std', 'count'])
season_stats['sem'] = season_stats['std'] / np.sqrt(season_stats['count'])
season_order = ['Winter', 'Spring', 'Summer', 'Fall']
season_stats = season_stats.reindex(season_order)

bars = ax.bar(range(4), season_stats['mean'],
              yerr=season_stats['sem'], capsize=8,
              color=['#5DADE2', '#58D68D', '#F4D03F', '#E67E22'],
              edgecolor='black', linewidth=2, alpha=0.8)
ax.set_xlabel('Season', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Seasonal PM2.5 Patterns', fontsize=12, fontweight='bold')
ax.set_xticks(range(4))
ax.set_xticklabels(season_order)
ax.grid(True, alpha=0.3, axis='y')

for i, (bar, val) in enumerate(zip(bars, season_stats['mean'])):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
           f'{val:.1f}', ha='center', fontsize=10, fontweight='bold')

ax = axes[0, 2]
monthly_by_year = df.groupby(['year', 'month'])['pm25_ugm3_mean'].mean().unstack(level=0)
for year in monthly_by_year.columns:
    ax.plot(monthly_by_year.index, monthly_by_year[year], marker='o', markersize=4, label=f'{year}', linewidth=2)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Monthly Patterns by Year', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names, rotation=45)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

ax = axes[1, 0]
monthly_ts = df.groupby(pd.Grouper(key='timestamp_jst', freq='M'))['pm25_ugm3_mean'].mean()
ax.plot(monthly_ts.index, monthly_ts.values, marker='o', linewidth=2, markersize=6, color='steelblue')
z = np.polyfit(range(len(monthly_ts)), monthly_ts.values, 1)
p = np.poly1d(z)
ax.plot(monthly_ts.index, p(range(len(monthly_ts))), "r--", alpha=0.8, linewidth=2, label=f'Trend: {z[0]:.3f} μg/m³/month')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Monthly Time Series with Trend', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend()
ax.tick_params(axis='x', rotation=45)

ax = axes[1, 1]
month_hour_matrix = df.groupby(['month', 'hour'])['pm25_ugm3_mean'].mean().unstack()
im = ax.imshow(month_hour_matrix, aspect='auto', cmap='YlOrRd', interpolation='bilinear')
ax.set_xlabel('Hour of Day (JST)', fontsize=11)
ax.set_ylabel('Month', fontsize=11)
ax.set_title('Month-Hour PM2.5 Heatmap (JST)', fontsize=12, fontweight='bold')
ax.set_xticks(range(0, 24, 3))
ax.set_yticks(range(12))
ax.set_yticklabels(month_names)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('PM2.5 (μg/m³)', rotation=270, labelpad=15)

ax = axes[1, 2]
month_temp = df.groupby('month')[['pm25_ugm3_mean', 'temperature_c_mean']].mean()
ax2 = ax.twinx()
l1 = ax.plot(month_temp.index, month_temp['pm25_ugm3_mean'], 'b-o', linewidth=2, markersize=6, label='PM2.5')
l2 = ax2.plot(month_temp.index, month_temp['temperature_c_mean'], 'r-s', linewidth=2, markersize=6, label='Temperature')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', color='b', fontsize=11)
ax2.set_ylabel('Temperature (°C)', color='r', fontsize=11)
ax.tick_params(axis='y', labelcolor='b')
ax2.tick_params(axis='y', labelcolor='r')
ax.set_title('PM2.5 vs Temperature by Month', fontsize=12, fontweight='bold')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names, rotation=45)
ax.grid(True, alpha=0.3)
lines = l1 + l2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right')

plt.suptitle('Monthly and Seasonal PM2.5 Patterns', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graphs/temporal/monthly_seasonal_patterns.png', dpi=300, bbox_inches='tight')
print("Saved: graphs/temporal/monthly_seasonal_patterns.png")
plt.show()

## 4. Long-term Trends

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

daily_avg = df.groupby('date')['pm25_ugm3_mean'].mean()
rolling_7d = daily_avg.rolling(window=7, center=True).mean()
rolling_30d = daily_avg.rolling(window=30, center=True).mean()

ax = axes[0, 0]
ax.plot(daily_avg.index, daily_avg.values, alpha=0.2, linewidth=0.5, color='gray', label='Daily')
ax.plot(rolling_7d.index, rolling_7d.values, linewidth=1.5, color='blue', label='7-day MA', alpha=0.7)
ax.plot(rolling_30d.index, rolling_30d.values, linewidth=2, color='red', label='30-day MA')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('PM2.5 Trend with Moving Averages', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)

for year in [2024, 2025]:
    year_start = pd.Timestamp(f'{year}-01-01')
    if year_start in daily_avg.index:
        ax.axvline(x=year_start, color='black', linestyle='--', alpha=0.3)
        ax.text(year_start, ax.get_ylim()[1]*0.95, str(year), fontsize=9)

ax = axes[0, 1]
yearly_avg = df.groupby('year')['pm25_ugm3_mean'].agg(['mean', 'std', 'count'])
yearly_avg['sem'] = yearly_avg['std'] / np.sqrt(yearly_avg['count'])
bars = ax.bar(yearly_avg.index, yearly_avg['mean'],
              yerr=yearly_avg['sem'], capsize=8,
              color=['#3498DB', '#E74C3C', '#2ECC71'],
              edgecolor='black', linewidth=2, alpha=0.8)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('Yearly Average PM2.5', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, yearly_avg['mean']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.1,
           f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

ax = axes[1, 0]
q05 = daily_avg.rolling(window=30, center=True).quantile(0.05)
q25 = daily_avg.rolling(window=30, center=True).quantile(0.25)
q50 = daily_avg.rolling(window=30, center=True).quantile(0.50)
q75 = daily_avg.rolling(window=30, center=True).quantile(0.75)
q95 = daily_avg.rolling(window=30, center=True).quantile(0.95)

ax.fill_between(q50.index, q05, q95, alpha=0.2, color='gray', label='5-95th percentile')
ax.fill_between(q50.index, q25, q75, alpha=0.3, color='steelblue', label='25-75th percentile')
ax.plot(q50.index, q50, linewidth=2, color='darkblue', label='Median')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('PM2.5 (μg/m³)', fontsize=11)
ax.set_title('PM2.5 Percentile Bands (30-day rolling)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right')
ax.tick_params(axis='x', rotation=45)

ax = axes[1, 1]
threshold = 15
exceedance = daily_avg > threshold
# Create proper datetime index for resampling
exceedance_series = pd.Series(exceedance.values, index=pd.DatetimeIndex(daily_avg.index))
monthly_exceedance = exceedance_series.resample('M').mean() * 100
ax.bar(monthly_exceedance.index, monthly_exceedance.values, 
       color='red', alpha=0.6, edgecolor='darkred', linewidth=1)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Days Exceeding WHO 24h Guideline (%)', fontsize=11)
ax.set_title(f'Monthly Exceedance Rate (>{threshold} μg/m³)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.tick_params(axis='x', rotation=45)
ax.axhline(y=10, color='orange', linestyle='--', alpha=0.5, label='10% threshold')
ax.legend()

plt.suptitle('Long-term PM2.5 Trends (2023-2025)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graphs/temporal/longterm_trends.png', dpi=300, bbox_inches='tight')
print("Saved: graphs/temporal/longterm_trends.png")
plt.show()

## Summary Statistics

In [ ]:
print("="*70)
print("PM2.5 TEMPORAL PATTERNS SUMMARY (Japan Standard Time)")
print("="*70)

print("\n📊 DAILY PATTERNS (JST):")
print(f"  Peak hour: {hourly_stats['mean'].idxmax()}:00 JST ({hourly_stats['mean'].max():.2f} μg/m³)")
print(f"  Lowest hour: {hourly_stats['mean'].idxmin()}:00 JST ({hourly_stats['mean'].min():.2f} μg/m³)")
print(f"  Daily range: {hourly_stats['mean'].max() - hourly_stats['mean'].min():.2f} μg/m³")
print(f"  Morning rush effect (8:00 JST): {(hourly_stats.loc[8, 'mean'] - hourly_stats['mean'].mean()):.2f} μg/m³ above average")

print("\n📅 WEEKLY PATTERNS:")
print(f"  Highest day: {dow_names[dow_stats['mean'].idxmax()]} ({dow_stats['mean'].max():.2f} μg/m³)")
print(f"  Lowest day: {dow_names[dow_stats['mean'].idxmin()]} ({dow_stats['mean'].min():.2f} μg/m³)")
print(f"  Weekend effect: {(weekend_comparison.loc[0, 'mean'] - weekend_comparison.loc[1, 'mean']):.2f} μg/m³ lower")
print(f"  Weekend reduction: {((weekend_comparison.loc[0, 'mean'] - weekend_comparison.loc[1, 'mean'])/weekend_comparison.loc[0, 'mean']*100):.1f}%")

print("\n📆 MONTHLY/SEASONAL:")
print(f"  Peak month: {month_names[monthly_stats['mean'].idxmax()-1]} ({monthly_stats['mean'].max():.2f} μg/m³)")
print(f"  Lowest month: {month_names[monthly_stats['mean'].idxmin()-1]} ({monthly_stats['mean'].min():.2f} μg/m³)")
print(f"  Seasonal range: {season_stats['mean'].max():.2f} - {season_stats['mean'].min():.2f} μg/m³")
print(f"  Highest season: {season_stats['mean'].idxmax()}")

print("\n📈 LONG-TERM TRENDS:")
print(f"  2023 average: {yearly_avg.loc[2023, 'mean']:.2f} μg/m³")
print(f"  2024 average: {yearly_avg.loc[2024, 'mean']:.2f} μg/m³")
print(f"  2025 average: {yearly_avg.loc[2025, 'mean']:.2f} μg/m³")
print(f"  Trend: {'+' if yearly_avg.loc[2025, 'mean'] > yearly_avg.loc[2023, 'mean'] else '-'}{abs(yearly_avg.loc[2025, 'mean'] - yearly_avg.loc[2023, 'mean']):.2f} μg/m³ from 2023 to 2025")

print("\n🔄 KEY TEMPORAL INSIGHTS:")
print("  • Morning rush hour peak clearly visible (7-9 AM JST)")
print("  • Weekend reduction effect confirmed (~5-10% lower)")
print("  • Strong seasonal patterns with winter/spring peaks")
print("  • Increasing trend in 2025 compared to 2023-2024")

print("\n" + "="*70)
print("All temporal pattern graphs saved to graphs/temporal/")
print("="*70)